In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Path to dataset files: /kaggle/input/skin-cancer-mnist-ham10000


In [2]:
"""
DINOv2 ViT-S/14 (frozen) + linear head on HAM10000.

Run on Kaggle with GPU + Internet enabled (needed once to pull DINOv2 weights).
Designed to be pasted into notebook cells — the CELL markers show the splits.

Why this shape:
  - Backbone is frozen, so features are extracted ONCE and cached to disk.
    Every experiment after that is a ~20 second head retrain, not a 40 min run.
  - Split is grouped on lesion_id. HAM10000 has multiple images per lesion;
    an image-level split leaks near-duplicates into val and lies to you.
  - Class weights are sqrt-inverse capped at 10:1. Raw inverse frequency on
    HAM is ~58:1 (nv 6705 vs df 115) and destabilises training.
"""


'\nDINOv2 ViT-S/14 (frozen) + linear head on HAM10000.\n \nRun on Kaggle with GPU + Internet enabled (needed once to pull DINOv2 weights).\nDesigned to be pasted into notebook cells — the CELL markers show the splits.\n \nWhy this shape:\n  - Backbone is frozen, so features are extracted ONCE and cached to disk.\n    Every experiment after that is a ~20 second head retrain, not a 40 min run.\n  - Split is grouped on lesion_id. HAM10000 has multiple images per lesion;\n    an image-level split leaks near-duplicates into val and lies to you.\n  - Class weights are sqrt-inverse capped at 10:1. Raw inverse frequency on\n    HAM is ~58:1 (nv 6705 vs df 115) and destabilises training.\n'

In [5]:

# ============================================================ CELL 1: setup
import os, glob, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from google.colab import drive
drive.mount("/content/drive")

SEED = 42
DATA = "/kaggle/input/skin-cancer-mnist-ham10000"
OUT = "/content/drive/MyDrive/amsdds"      # persists across Colab disconnects
os.makedirs(OUT, exist_ok=True)
IMG_SIZE = 224          # must be a multiple of 14 for ViT-S/14
BATCH = 64
AUG_COPIES = 2          # extra augmented feature copies for the TRAIN split only
USE_COLOR_CONSTANCY = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(SEED); np.random.seed(SEED)
print("device:", DEVICE)



Mounted at /content/drive
device: cuda


In [7]:

# ==================================================== CELL 2: metadata + split
from sklearn.model_selection import StratifiedGroupKFold

meta_path = glob.glob(f"{DATA}/**/HAM10000_metadata*.csv", recursive=True)[0]
df = pd.read_csv(meta_path)

# Folder names vary between dataset versions — just index every jpg we can find.
paths = {}
for p in glob.glob(f"{DATA}/**/*.jpg", recursive=True):
    paths[os.path.splitext(os.path.basename(p))[0]] = p
df["path"] = df.image_id.map(paths)
missing = df.path.isna().sum()
assert missing == 0, f"{missing} images not found under {DATA}"

CLASSES = sorted(df.dx.unique())            # akiec bcc bkl df mel nv vasc
C2I = {c: i for i, c in enumerate(CLASSES)}
df["y"] = df.dx.map(C2I)

# ~71 / 14.3 / 14.3, grouped by lesion so no lesion spans two splits
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
fold = np.zeros(len(df), dtype=int)
for f, (_, idx) in enumerate(sgkf.split(df, df.y, groups=df.lesion_id)):
    fold[idx] = f
df["split"] = np.where(fold == 0, "test", np.where(fold == 1, "val", "train"))

print(df.split.value_counts(), "\n")
print(pd.crosstab(df.dx, df.split))

# sanity: no lesion_id appears in more than one split
leak = df.groupby("lesion_id").split.nunique().max()
assert leak == 1, "lesion leaked across splits"
print("\nno lesion leakage across splits")

df.to_csv(f"{OUT}/splits.csv", index=False)


split
train    7116
val      1458
test     1441
Name: count, dtype: int64 

split  test  train  val
dx                     
akiec    34    230   63
bcc      89    358   67
bkl     168    772  159
df        9     89   17
mel     146    782  185
nv      974   4796  935
vasc     21     89   32

no lesion leakage across splits


In [8]:

# ============================================== CELL 3: dataset + transforms
def shades_of_gray(arr, power=6):
    """Illumination normalisation (Barata et al.). The standard dermoscopy
    colour-constancy trick — used by the ISIC 2019 winners. Cheap and it
    directly targets the lighting-variation failure mode."""
    a = arr.astype(np.float32)
    vec = np.power(np.mean(np.power(a, power), axis=(0, 1)), 1.0 / power)
    vec = vec / (np.sqrt(np.sum(vec ** 2)) + 1e-8)
    a = a / (vec * np.sqrt(3) + 1e-8)
    return np.clip(a, 0, 255).astype(np.uint8)


MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

eval_tf = T.Compose([
    T.Resize(256), T.CenterCrop(IMG_SIZE),
    T.ToTensor(), T.Normalize(MEAN, STD),
])

# Augmentations lean on photometric jitter because lighting/quality robustness
# is the stated requirement — not just geometric flips.
aug_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0), ratio=(0.85, 1.18)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomApply([T.RandomRotation(30)], p=0.5),
    T.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.25, hue=0.03),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.25),
    T.ToTensor(), T.Normalize(MEAN, STD),
])


class HAM(Dataset):
    def __init__(self, frame, tf, cc=USE_COLOR_CONSTANCY):
        self.f = frame.reset_index(drop=True); self.tf = tf; self.cc = cc

    def __len__(self):
        return len(self.f)

    def __getitem__(self, i):
        r = self.f.iloc[i]
        img = Image.open(r.path).convert("RGB")
        if self.cc:
            img = Image.fromarray(shades_of_gray(np.array(img)))
        return self.tf(img), int(r.y)



In [9]:

# ================================================= CELL 4: DINOv2 backbone
import timm

# reg4 variant (with registers) is a little cleaner; drop "_reg4" if it 404s.
MODEL_NAME = "vit_small_patch14_reg4_dinov2.lvd142m"
backbone = timm.create_model(
    MODEL_NAME, pretrained=True, num_classes=0, img_size=IMG_SIZE
).eval().to(DEVICE)
for p in backbone.parameters():
    p.requires_grad_(False)

NPREFIX = backbone.num_prefix_tokens
FEAT_DIM = backbone.embed_dim * 2   # CLS ++ mean(patch tokens) = 768
print(f"{MODEL_NAME} | prefix tokens {NPREFIX} | feature dim {FEAT_DIM}")


@torch.no_grad()
def embed(loader):
    """CLS token concatenated with the mean patch token — the DINOv2 paper's
    own linear-probe recipe, and meaningfully better than CLS alone."""
    fs, ys = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            tok = backbone.forward_features(x)
        cls = tok[:, 0]
        patch = tok[:, NPREFIX:].mean(dim=1)
        fs.append(torch.cat([cls, patch], dim=1).float().cpu())
        ys.append(y)
    return torch.cat(fs).numpy(), torch.cat(ys).numpy()



model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

vit_small_patch14_reg4_dinov2.lvd142m | prefix tokens 5 | feature dim 768


In [10]:

# ============================================== CELL 5: extract + cache once
def loader_for(frame, tf, shuffle=False):
    return DataLoader(HAM(frame, tf), batch_size=BATCH, shuffle=shuffle,
                      num_workers=2, pin_memory=True)


cache = f"{OUT}/dinov2_feats.npz"
if os.path.exists(cache):
    z = np.load(cache)
    Xtr, ytr = z["Xtr"], z["ytr"]
    Xva, yva = z["Xva"], z["yva"]
    Xte, yte = z["Xte"], z["yte"]
    print("loaded cached features")
else:
    t0 = time.time()
    tr, va, te = (df[df.split == s] for s in ("train", "val", "test"))

    Xtr, ytr = embed(loader_for(tr, eval_tf))
    for k in range(AUG_COPIES):                     # augmented copies of train
        Xa, ya = embed(loader_for(tr, aug_tf))
        Xtr = np.concatenate([Xtr, Xa]); ytr = np.concatenate([ytr, ya])
        print(f"  aug copy {k+1}/{AUG_COPIES} done")

    Xva, yva = embed(loader_for(va, eval_tf))
    Xte, yte = embed(loader_for(te, eval_tf))

    np.savez_compressed(cache, Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva, Xte=Xte, yte=yte)
    print(f"extracted in {time.time()-t0:.0f}s | train {Xtr.shape} val {Xva.shape} test {Xte.shape}")



  aug copy 1/2 done
  aug copy 2/2 done
extracted in 664s | train (21348, 768) val (1458, 768) test (1441, 768)


In [11]:

# ============================================== CELL 6: train the linear head
# Standardising features makes the linear head converge far faster.
mu, sd = Xtr.mean(0, keepdims=True), Xtr.std(0, keepdims=True) + 1e-6
ztr, zva, zte = ((X - mu) / sd for X in (Xtr, Xva, Xte))

# sqrt-inverse frequency, capped at 10:1
counts = np.bincount(ytr, minlength=len(CLASSES)).astype(np.float32)
w = 1.0 / np.sqrt(counts)
w = w / w.min()
w = np.clip(w, 1.0, 10.0)
print("class weights:", dict(zip(CLASSES, np.round(w, 2))))

tX = torch.tensor(ztr, dtype=torch.float32).to(DEVICE)
tY = torch.tensor(ytr, dtype=torch.long).to(DEVICE)
vX = torch.tensor(zva, dtype=torch.float32).to(DEVICE)
vY = torch.tensor(yva, dtype=torch.long).to(DEVICE)

head = nn.Linear(FEAT_DIM, len(CLASSES)).to(DEVICE)
opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS = 40
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
lossfn = nn.CrossEntropyLoss(
    weight=torch.tensor(w).to(DEVICE),
    label_smoothing=0.05,          # curbs overconfidence at the source
)

from sklearn.metrics import f1_score

best_f1, best_state = -1, None
for ep in range(EPOCHS):
    head.train()
    perm = torch.randperm(len(tX), device=DEVICE)
    for i in range(0, len(tX), 512):
        idx = perm[i:i + 512]
        opt.zero_grad()
        lossfn(head(tX[idx]), tY[idx]).backward()
        opt.step()
    sched.step()

    head.eval()
    with torch.no_grad():
        pv = head(vX).argmax(1).cpu().numpy()
    f1 = f1_score(yva, pv, average="macro")
    if f1 > best_f1:
        best_f1 = f1
        best_state = {k: v.clone() for k, v in head.state_dict().items()}
    if ep % 5 == 0 or ep == EPOCHS - 1:
        print(f"ep {ep:2d}  val macro-F1 {f1:.4f}")

head.load_state_dict(best_state)
print(f"\nbest val macro-F1 {best_f1:.4f}")



class weights: {'akiec': np.float32(4.57), 'bcc': np.float32(3.66), 'bkl': np.float32(2.49), 'df': np.float32(7.34), 'mel': np.float32(2.48), 'nv': np.float32(1.0), 'vasc': np.float32(7.34)}
ep  0  val macro-F1 0.4650
ep  5  val macro-F1 0.5860
ep 10  val macro-F1 0.5999
ep 15  val macro-F1 0.6303
ep 20  val macro-F1 0.6362
ep 25  val macro-F1 0.6565
ep 30  val macro-F1 0.6560
ep 35  val macro-F1 0.6593
ep 39  val macro-F1 0.6593

best val macro-F1 0.6605


In [12]:

# ================================ CELL 7: temperature scaling on the val split
logit_va = head(vX).detach()
logT = torch.zeros(1, device=DEVICE, requires_grad=True)
topt = torch.optim.LBFGS([logT], lr=0.1, max_iter=60)


def _closure():
    topt.zero_grad()
    l = F.cross_entropy(logit_va / logT.exp(), vY)
    l.backward()
    return l


topt.step(_closure)
TEMP = float(logT.exp())
print(f"fitted temperature T = {TEMP:.3f}   (T>1 means the head was overconfident)")



fitted temperature T = 0.678   (T>1 means the head was overconfident)


/tmp/ipykernel_1039/2953847154.py:15: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  TEMP = float(logT.exp())


In [13]:

# ==================================================== CELL 8: evaluate on test
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


def ece(probs, labels, bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    acc = (pred == labels).astype(np.float32)
    edges = np.linspace(0, 1, bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e


head.eval()
with torch.no_grad():
    logits = head(torch.tensor(zte, dtype=torch.float32).to(DEVICE))
    p_raw = F.softmax(logits, 1).cpu().numpy()
    p_cal = F.softmax(logits / TEMP, 1).cpu().numpy()
pred = p_cal.argmax(1)

print(f"\ntest accuracy   {accuracy_score(yte, pred):.4f}")
print(f"test macro-F1   {f1_score(yte, pred, average='macro'):.4f}")
print(f"ECE  raw {ece(p_raw, yte):.4f}  ->  calibrated {ece(p_cal, yte):.4f}")
print("\n", classification_report(yte, pred, target_names=CLASSES, digits=3))
print(pd.DataFrame(confusion_matrix(yte, pred), index=CLASSES, columns=CLASSES))

torch.save({"head": head.state_dict(), "mu": mu, "sd": sd,
            "temp": TEMP, "classes": CLASSES, "model_name": MODEL_NAME},
           f"{OUT}/layer1_dinov2_head.pt")




test accuracy   0.7446
test macro-F1   0.5545
ECE  raw 0.1317  ->  calibrated 0.0204

               precision    recall  f1-score   support

       akiec      0.291     0.471     0.360        34
         bcc      0.633     0.697     0.663        89
         bkl      0.573     0.631     0.601       168
          df      0.182     0.444     0.258         9
         mel      0.360     0.548     0.435       146
          nv      0.950     0.808     0.873       974
        vasc      0.581     0.857     0.692        21

    accuracy                          0.745      1441
   macro avg      0.510     0.637     0.555      1441
weighted avg      0.801     0.745     0.766      1441

       akiec  bcc  bkl  df  mel   nv  vasc
akiec     16    8    5   0    5    0     0
bcc        8   62    5   4   10    0     0
bkl       14    8  106   5   22   13     0
df         4    0    0   4    0    1     0
mel        6    6   24   1   80   27     2
nv         7   13   45   8  103  787    11
vasc       0  

In [14]:

# ======================= CELL 9: OOD scores, free once features already exist
# Energy is one line and beats max-softmax. Mahalanobis uses the frozen feature
# space you already have. Both give the Decision Engine a better gate signal
# than softmax confidence, which Guo et al. showed you shouldn't trust.
with torch.no_grad():
    energy = (-torch.logsumexp(logits / TEMP, dim=1)).cpu().numpy()

means, prec = [], None
Sw = np.zeros((FEAT_DIM, FEAT_DIM), dtype=np.float64)
for c in range(len(CLASSES)):
    fc = ztr[ytr == c]
    m = fc.mean(0); means.append(m)
    d = fc - m
    Sw += d.T @ d
prec = np.linalg.pinv(Sw / len(ztr))
means = np.stack(means)


def mahalanobis(Z):
    d = Z[:, None, :] - means[None]                      # (N, C, D)
    return np.einsum("ncd,de,nce->nc", d, prec, d).min(1)


maha = mahalanobis(zte)
np.savez(f"{OUT}/ood_scores.npz", energy=energy, maha=maha, y=yte,
         conf=p_cal.max(1), means=means, prec=prec)
print(f"\nOOD scores saved. energy mean {energy.mean():.3f} | maha mean {maha.mean():.1f}")
print("Next: hold one class out of training entirely, then check whether these "
      "scores separate it from the six seen classes. That is your unknown-disease number.")


OOD scores saved. energy mean -3.461 | maha mean 855.7
Next: hold one class out of training entirely, then check whether these scores separate it from the six seen classes. That is your unknown-disease number.


In [15]:
from sklearn.metrics import f1_score

def train_head(cap, hidden=512, epochs=60, lr=2e-3):
    counts = np.bincount(ytr, minlength=len(CLASSES)).astype(np.float32)
    w = 1.0 / np.sqrt(counts); w = np.clip(w / w.min(), 1.0, cap)
    head = nn.Sequential(
        nn.LayerNorm(FEAT_DIM),
        nn.Linear(FEAT_DIM, hidden), nn.GELU(), nn.Dropout(0.3),
        nn.Linear(hidden, len(CLASSES)),
    ).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    lossfn = nn.CrossEntropyLoss(weight=torch.tensor(w).to(DEVICE), label_smoothing=0.05)
    best, state = -1, None
    for ep in range(epochs):
        head.train()
        perm = torch.randperm(len(tX), device=DEVICE)
        for i in range(0, len(tX), 512):
            idx = perm[i:i+512]
            opt.zero_grad(); lossfn(head(tX[idx]), tY[idx]).backward(); opt.step()
        sched.step()
        head.eval()
        with torch.no_grad(): pv = head(vX).argmax(1).cpu().numpy()
        f1 = f1_score(yva, pv, average="macro")
        if f1 > best: best, state = f1, {k: v.clone() for k, v in head.state_dict().items()}
    head.load_state_dict(state)
    return head, best

results = {}
for cap in [1.0, 2.0, 3.0, 5.0, 10.0]:
    h, f1 = train_head(cap)
    results[cap] = (h, f1)
    print(f"cap {cap:4.1f}  val macro-F1 {f1:.4f}")

CAP = max(results, key=lambda c: results[c][1])
head = results[CAP][0]
print(f"\nbest cap {CAP}  val macro-F1 {results[CAP][1]:.4f}")

cap  1.0  val macro-F1 0.6908
cap  2.0  val macro-F1 0.7174
cap  3.0  val macro-F1 0.7036
cap  5.0  val macro-F1 0.7057
cap 10.0  val macro-F1 0.7015

best cap 2.0  val macro-F1 0.7174


In [16]:

# ================================ CELL 7: temperature scaling on the val split
logit_va = head(vX).detach()
logT = torch.zeros(1, device=DEVICE, requires_grad=True)
topt = torch.optim.LBFGS([logT], lr=0.1, max_iter=60)


def _closure():
    topt.zero_grad()
    l = F.cross_entropy(logit_va / logT.exp(), vY)
    l.backward()
    return l


topt.step(_closure)
TEMP = float(logT.exp())
print(f"fitted temperature T = {TEMP:.3f}   (T>1 means the head was overconfident)")



fitted temperature T = 0.938   (T>1 means the head was overconfident)


In [17]:

# ==================================================== CELL 8: evaluate on test
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


def ece(probs, labels, bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    acc = (pred == labels).astype(np.float32)
    edges = np.linspace(0, 1, bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e


head.eval()
with torch.no_grad():
    logits = head(torch.tensor(zte, dtype=torch.float32).to(DEVICE))
    p_raw = F.softmax(logits, 1).cpu().numpy()
    p_cal = F.softmax(logits / TEMP, 1).cpu().numpy()
pred = p_cal.argmax(1)

print(f"\ntest accuracy   {accuracy_score(yte, pred):.4f}")
print(f"test macro-F1   {f1_score(yte, pred, average='macro'):.4f}")
print(f"ECE  raw {ece(p_raw, yte):.4f}  ->  calibrated {ece(p_cal, yte):.4f}")
print("\n", classification_report(yte, pred, target_names=CLASSES, digits=3))
print(pd.DataFrame(confusion_matrix(yte, pred), index=CLASSES, columns=CLASSES))

torch.save({"head": head.state_dict(), "mu": mu, "sd": sd,
            "temp": TEMP, "classes": CLASSES, "model_name": MODEL_NAME},
           f"{OUT}/layer1_dinov2_head.pt")




test accuracy   0.8119
test macro-F1   0.6387
ECE  raw 0.0251  ->  calibrated 0.0183

               precision    recall  f1-score   support

       akiec      0.344     0.324     0.333        34
         bcc      0.800     0.719     0.757        89
         bkl      0.626     0.637     0.631       168
          df      0.571     0.444     0.500         9
         mel      0.461     0.521     0.489       146
          nv      0.921     0.916     0.918       974
        vasc      0.941     0.762     0.842        21

    accuracy                          0.812      1441
   macro avg      0.666     0.617     0.639      1441
weighted avg      0.817     0.812     0.814      1441

       akiec  bcc  bkl  df  mel   nv  vasc
akiec     11    5    9   0    8    1     0
bcc        4   64    9   1    9    2     0
bkl       10    0  107   2   18   31     0
df         3    0    0   4    0    2     0
mel        3    3   24   0   76   40     0
nv         1    7   22   0   51  892     1
vasc       0  

In [18]:
ckpt = {
    "head_state": head.state_dict(),
    "arch": {"feat_dim": FEAT_DIM, "hidden": 512, "dropout": 0.3,
             "n_classes": len(CLASSES)},
    "feat_mu": mu, "feat_sd": sd,          # feature standardisation stats
    "temperature": TEMP,
    "classes": CLASSES,
    "backbone": MODEL_NAME,
    "img_size": IMG_SIZE,
    "prefix_tokens": NPREFIX,
    "color_constancy": USE_COLOR_CONSTANCY,
    "weight_cap": CAP,
    "metrics": {"test_acc": 0.8119, "test_macro_f1": 0.6387,
                "ece_raw": 0.0251, "ece_cal": 0.0183, "val_macro_f1": 0.7174},
}
torch.save(ckpt, f"{OUT}/layer1_dinov2_v1.pt")
print("saved:", f"{OUT}/layer1_dinov2_v1.pt")

saved: /content/drive/MyDrive/amsdds/layer1_dinov2_v1.pt
